# Task 1C: Feature Engineering

## Approach
Following the advanced assignment instructions and Figure 1, we transform the time series into a **sliding-window instance-based dataset**.

For each participant and each day `t`, we aggregate the preceding `W` days (the history window) into a flat feature vector, and set the **target** as the average mood on day `t`.

### Feature categories we engineer (inspired by literature):
| Category | Features | Rationale |
|---|---|---|
| **Lag statistics** | mean, std, min, max of each variable over the window | Capture level and variability (Chikersal et al., 2021) |
| **Mood trend** | linear slope of mood over window | Direction of mood change matters (Canzian & Musolesi, 2015) |
| **Recent vs. distal mood** | mean mood last 2 days vs. days 3–W | Recency weighting captures inertia (Wang et al., 2014) |
| **Screen & app ratios** | social / total screen, entertainment / total screen | Relative usage more informative than absolute (Montag et al., 2019) |
| **Circadian rhythm proxy** | rolling std of activity | Irregular activity patterns linked to mood disorders (Saeb et al., 2015) |
| **Target** | mean mood on day `t` | Per assignment spec |

Window size `W = 7` days — a natural weekly cycle, widely used in mental health sensing literature.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Load cleaned daily data (output of Task 1B) ───────────────────────────
daily = pd.read_csv('daily_cleaned.csv', parse_dates=['date'])
daily = daily.sort_values(['id', 'date']).reset_index(drop=True)

print(f'Loaded daily_cleaned.csv: {daily.shape}')
print(f'Participants: {daily["id"].nunique()}')
print(f'Date range: {daily["date"].min().date()} → {daily["date"].max().date()}')
daily.head()

In [ ]:
# ── Define feature columns (all except id, date, mood) ───────────────────
EXCLUDE = {'id', 'date', 'mood'}
sensor_cols = [c for c in daily.columns if c not in EXCLUDE]

WINDOW = 7   # days of history used to predict mood on day t

print(f'Window size W = {WINDOW} days')
print(f'Sensor columns ({len(sensor_cols)}): {sensor_cols}')

## Helper: slope of a short time series
We use ordinary least squares slope as a simple trend indicator.

In [ ]:
def ols_slope(arr):
    """Return OLS slope of arr vs. integer time index. NaN if <2 valid points."""
    arr = np.asarray(arr, dtype=float)
    valid = ~np.isnan(arr)
    if valid.sum() < 2:
        return np.nan
    x = np.where(valid)[0]
    y = arr[valid]
    slope, *_ = np.polyfit(x, y, 1)
    return slope

## Build the instance-based dataset

For each participant, we slide a window of `W` days and create one training instance per day `t ≥ W`.  
The instance contains:
- Window aggregates of all sensor variables
- Mood-specific engineered features (trend, recency split)
- Ratio features (screen usage composition)
- The target: average mood on day `t`

In [ ]:
instances = []

for pid, grp in daily.groupby('id'):
    grp = grp.reset_index(drop=True)
    n = len(grp)

    for t in range(WINDOW, n):           # t is the prediction day index
        window = grp.iloc[t - WINDOW: t]  # W days of history
        target_mood = grp.loc[t, 'mood']  # mood ON day t (the target)

        if pd.isna(target_mood):
            continue   # skip days without a mood label

        row = {'id': pid, 'date': grp.loc[t, 'date'], 'target_mood': target_mood}

        # ── 1. Window statistics for every sensor column ──────────────────
        for col in sensor_cols:
            vals = window[col].to_numpy(dtype=float)
            row[f'{col}_mean'] = np.nanmean(vals)
            row[f'{col}_std']  = np.nanstd(vals)
            row[f'{col}_min']  = np.nanmin(vals)
            row[f'{col}_max']  = np.nanmax(vals)

        # ── 2. Mood trend (OLS slope over the window) ─────────────────────
        row['mood_slope'] = ols_slope(window['mood'].to_numpy())

        # ── 3. Recent vs. distal mood (last 2 days vs. days 3–W) ─────────
        recent  = window['mood'].iloc[-2:].to_numpy(dtype=float)
        distal  = window['mood'].iloc[:-2].to_numpy(dtype=float)
        row['mood_recent_mean']  = np.nanmean(recent)
        row['mood_distal_mean']  = np.nanmean(distal)
        row['mood_recency_diff'] = row['mood_recent_mean'] - row['mood_distal_mean']

        # ── 4. Screen / app ratio features ────────────────────────────────
        app_cols = [c for c in sensor_cols if c.startswith('appCat')]
        total_screen = np.nansum([np.nanmean(window[c].to_numpy(dtype=float)) for c in app_cols])

        for cat in ['appCat.social', 'appCat.entertainment', 'appCat.communication']:
            if cat in daily.columns:
                cat_mean = np.nanmean(window[cat].to_numpy(dtype=float))
                ratio = cat_mean / total_screen if total_screen > 0 else 0.0
                row[f'{cat}_ratio'] = ratio

        # ── 5. Circadian rhythm proxy (std of activity across window) ─────
        if 'activity' in daily.columns:
            row['activity_circadian_std'] = np.nanstd(window['activity'].to_numpy(dtype=float))

        instances.append(row)

df_instances = pd.DataFrame(instances).reset_index(drop=True)
print(f'Instance-based dataset shape: {df_instances.shape}')
print(f'Features: {df_instances.shape[1] - 3} (excl. id, date, target)')
df_instances.head()

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────
print('Instances per participant:')
print(df_instances.groupby('id').size().describe())
print(f'\nTarget (mood) distribution:')
print(df_instances['target_mood'].describe().round(3))
print(f'\nOverall NaN rate in features: {df_instances.drop(columns=["id","date","target_mood"]).isna().mean().mean()*100:.1f}%')

## Visualisation 1 – Target distribution
The target is the **daily average mood** (1–10 scale). Understanding its distribution is important for choosing the right modelling approach.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of target mood
axes[0].hist(df_instances['target_mood'].dropna(), bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of target mood (daily average)')
axes[0].set_xlabel('Mood')
axes[0].set_ylabel('Count')

# Mood slope distribution
axes[1].hist(df_instances['mood_slope'].dropna(), bins=30, color='tomato', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', lw=1)
axes[1].set_title('Distribution of mood trend (OLS slope over 7-day window)')
axes[1].set_xlabel('Slope')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150)
plt.show()

## Visualisation 2 – Feature correlations with target mood
We inspect which engineered features correlate most strongly with the target to validate that the feature engineering is informative.

In [ ]:
feat_cols = [c for c in df_instances.columns if c not in ('id', 'date', 'target_mood')]

correlations = (
    df_instances[feat_cols + ['target_mood']]
    .corr()['target_mood']
    .drop('target_mood')
    .dropna()
    .sort_values(key=abs, ascending=False)
)

top20 = correlations.head(20)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'tomato' for v in top20.values]
plt.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1])
plt.axvline(0, color='black', lw=0.8)
plt.title('Top 20 feature correlations with target mood')
plt.xlabel('Pearson r')
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=150)
plt.show()

print('Top 10 correlated features:')
print(top20.head(10).round(3))

## Visualisation 3 – Recency effect on mood
We validate our `mood_recency_diff` feature: do instances where recent mood > distal mood tend to have higher target mood?

In [ ]:
df_plot = df_instances[['mood_recency_diff', 'target_mood', 'mood_slope']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df_plot['mood_recency_diff'], df_plot['target_mood'],
                alpha=0.25, s=8, color='steelblue')
m, b = np.polyfit(df_plot['mood_recency_diff'], df_plot['target_mood'], 1)
x_line = np.linspace(df_plot['mood_recency_diff'].min(), df_plot['mood_recency_diff'].max(), 100)
axes[0].plot(x_line, m * x_line + b, 'r-', lw=1.5)
axes[0].set_xlabel('Recency diff (recent − distal mood mean)')
axes[0].set_ylabel('Target mood')
axes[0].set_title('Recency effect on next-day mood')

axes[1].scatter(df_plot['mood_slope'], df_plot['target_mood'],
                alpha=0.25, s=8, color='tomato')
m2, b2 = np.polyfit(df_plot['mood_slope'], df_plot['target_mood'], 1)
x_line2 = np.linspace(df_plot['mood_slope'].min(), df_plot['mood_slope'].max(), 100)
axes[1].plot(x_line2, m2 * x_line2 + b2, 'b-', lw=1.5)
axes[1].set_xlabel('Mood OLS slope (7-day window)')
axes[1].set_ylabel('Target mood')
axes[1].set_title('Mood trend vs. next-day mood')

plt.tight_layout()
plt.savefig('recency_and_trend.png', dpi=150)
plt.show()

## Visualisation 4 – Instances per participant (dataset size check)
Since we lose the first `W=7` days per participant, we check no participant is left with too few instances.

In [ ]:
counts = df_instances.groupby('id').size().sort_values()

plt.figure(figsize=(12, 4))
counts.plot(kind='bar', color='steelblue')
plt.axhline(WINDOW, color='red', linestyle='--', label=f'Window size (W={WINDOW})')
plt.title('Number of training instances per participant')
plt.xlabel('Participant')
plt.ylabel('Instances')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('instances_per_participant.png', dpi=150)
plt.show()

print(f'Participants with < 14 instances: {(counts < 14).sum()}')

## Save the final feature-engineered dataset

In [ ]:
df_instances.to_csv('features_engineered.csv', index=False)
print(f'Saved: features_engineered.csv')
print(f'Shape: {df_instances.shape}')
print(f'\nFeature groups:')
print(f'  Window stats (mean/std/min/max × {len(sensor_cols)} sensors): {len(sensor_cols)*4}')
print(f'  Mood trend features (slope, recency diff, recent/distal mean): 4')
print(f'  Screen ratio features: {len([c for c in df_instances.columns if "ratio" in c])}')
print(f'  Circadian proxy: 1')
print(f'  Total features: {len(feat_cols)}')